# ERA5-Land — Runoff (ro, sro, ssro)

Visualize the consolidated ERA5-Land monthly runoff dataset from ECMWF
(Copernicus CDS, `reanalysis-era5-land`).

> **Note:** The fetch is still running; the monthly file currently accessible
> covers only the years completed so far.  The notebook adapts to whatever
> data are present.

- **Variable:** `ro` — total runoff (m per month)
- **Also stored:** `sro` (surface runoff), `ssro` (sub-surface runoff / recharge proxy)
- **Resolution:** 0.1° × 0.1° CONUS + contributing watersheds
- **Period:** 1979–present (monthly, fetch in progress)
- **Reference:** Muñoz-Sabater et al., 2021, doi:10.5194/essd-13-4349-2021

In [ ]:
from pathlib import Path
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

# --- Configuration ---
DATASTORE = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/nhf-datastore")
PROJECT   = Path("/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets")

# Pick the latest completed monthly file (rolling filename)
monthly_files = sorted(glob(str(DATASTORE / "era5_land" / "monthly" / "era5_land_monthly_*.nc")))
if not monthly_files:
    raise FileNotFoundError("No era5_land monthly files found — is the fetch still in progress?")
NC_PATH = Path(monthly_files[-1])
print(f"Using: {NC_PATH.name}")

PRIMARY_VAR = "ro"   # total runoff (m)


In [ ]:
ds = xr.open_dataset(NC_PATH)
print(ds)
print(f"\nTime range: {ds.time.values[0]} to {ds.time.values[-1]}")
print(f"Time steps: {ds.time.size}")
print(f"Grid:       {ds.lat.size} lat x {ds.lon.size} lon")
print(f"Variables:  {list(ds.data_vars)}")
print(f"Conventions: {ds.attrs.get('Conventions', 'N/A')}")


## Single-month total runoff map

In [ ]:
# Use the most recent available timestep
target_time = ds.time.values[-1]
da = ds[PRIMARY_VAR].sel(time=target_time)
actual_time = str(da.time.values)[:10]

fig, ax = plt.subplots(figsize=(14, 6))
da.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "m"})
ax.set_title(f"ERA5-Land Total Runoff (ro) — {actual_time}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"Stats for {actual_time}:")
print(f"  min:  {float(da.min(skipna=True)):.5f} m")
print(f"  max:  {float(da.max(skipna=True)):.5f} m")
print(f"  mean: {float(da.mean(skipna=True)):.5f} m")
print(f"  NaN%: {float(da.isnull().mean()) * 100:.1f}%")


## All three runoff variables — single timestep

In [ ]:
run_vars = {
    "ro":   "Total runoff (m)",
    "sro":  "Surface runoff (m)",
    "ssro": "Sub-surface runoff / recharge proxy (m)",
}

fig, axes = plt.subplots(1, 3, figsize=(20, 5))

for ax, (var, label) in zip(axes, run_vars.items()):
    if var not in ds.data_vars:
        ax.set_title(f"{label}\n(not in dataset)")
        ax.set_visible(False)
        continue
    da = ds[var].sel(time=target_time)
    da.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "m"})
    ax.set_title(f"{var}\n{label}")
    ax.set_aspect("equal")

fig.suptitle(f"ERA5-Land Runoff Variables — {actual_time}", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


## CONUS subset — mean over available data

In [ ]:
# ERA5-Land lat is descending (53→24.7); slice(upper, lower) selects CONUS
conus = ds[PRIMARY_VAR].sel(
    lat=slice(50.1, 23.9),
    lon=slice(-125.1, -65.9),
)

mean_runoff = conus.mean(dim="time")
start_yr = str(ds.time.values[0])[:4]
end_yr   = str(ds.time.values[-1])[:4]

fig, ax = plt.subplots(figsize=(14, 8))
mean_runoff.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "m"})
ax.set_title(f"Mean Total Runoff (ro) — CONUS {start_yr}–{end_yr}")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

print(f"CONUS {start_yr}–{end_yr} mean stats:")
print(f"  min:  {float(mean_runoff.min(skipna=True)):.5f} m")
print(f"  max:  {float(mean_runoff.max(skipna=True)):.5f} m")
print(f"  mean: {float(mean_runoff.mean(skipna=True)):.5f} m")


## Seasonal comparison (CONUS, available data)

In [ ]:
seasons = {
    "DJF (Winter)": conus.sel(time=conus.time.dt.month.isin([12, 1, 2])),
    "MAM (Spring)": conus.sel(time=conus.time.dt.month.isin([3, 4, 5])),
    "JJA (Summer)": conus.sel(time=conus.time.dt.month.isin([6, 7, 8])),
    "SON (Fall)":   conus.sel(time=conus.time.dt.month.isin([9, 10, 11])),
}

fig, axes = plt.subplots(2, 2, figsize=(16, 12))

for ax, (label, seasonal_data) in zip(axes.flatten(), seasons.items()):
    if seasonal_data.time.size == 0:
        ax.set_title(f"{label}\n(no data available)")
        continue
    seasonal_mean = seasonal_data.mean(dim="time")
    seasonal_mean.plot(ax=ax, cmap="YlGnBu", cbar_kwargs={"label": "m"})
    ax.set_title(f"{label} mean")
    ax.set_aspect("equal")

fig.suptitle(f"Seasonal Mean Total Runoff (ro) — CONUS {start_yr}–{end_yr}", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()


## Monthly time series (CONUS spatial mean)

In [ ]:
conus_full = ds[PRIMARY_VAR].sel(
    lat=slice(50.1, 23.9),
    lon=slice(-125.1, -65.9),
)
monthly_mean = conus_full.mean(dim=["lat", "lon"])

fig, ax = plt.subplots(figsize=(16, 4))
monthly_mean.plot(ax=ax, color="#2980b9", linewidth=0.8)
ax.set_ylabel("Mean Total Runoff (m)")
ax.set_title("CONUS-mean monthly total runoff (ro) — ERA5-Land")
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()


## Monthly climatology (CONUS, available data)

In [ ]:
monthly_clim = conus.groupby("time.month").mean(dim=["time", "lat", "lon"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(monthly_clim.month, monthly_clim.values, color="#2980b9", edgecolor="white")
ax.set_xlabel("Month")
ax.set_ylabel("Mean Total Runoff (m)")
ax.set_title(f"Monthly Climatology — CONUS {start_yr}–{end_yr} (ro)")
ax.set_xticks(range(1, 13))
ax.set_xticklabels(["J", "F", "M", "A", "M", "J", "J", "A", "S", "O", "N", "D"])
ax.set_ylim(bottom=0)
plt.tight_layout()
plt.show()


## CF compliance verification

In [ ]:
print("CF Compliance Checks:")
print(f"  Conventions attr:  {ds.attrs.get('Conventions', 'MISSING')}")
print(f"  Time dtype:        {ds.time.dtype}")
print(f"  Time first:        {ds.time.values[0]}")
print(f"  Time last:         {ds.time.values[-1]}")
print(f"  CRS variable:      {'crs' in ds.data_vars}")
if "crs" in ds.data_vars:
    print(f"  Grid mapping name: {ds['crs'].attrs.get('grid_mapping_name', 'MISSING')}")
    crs_wkt = ds['crs'].attrs.get('crs_wkt', 'MISSING')
    print(f"  CRS WKT:           {str(crs_wkt)[:60]}...")
print(f"  time_bnds:         {'time_bnds' in ds.data_vars}")
for var in ["ro", "sro", "ssro"]:
    if var in ds.data_vars:
        print(f"  {var}:")
        print(f"    grid_mapping: {ds[var].attrs.get('grid_mapping', 'MISSING')}")
        print(f"    units:        {ds[var].attrs.get('units', 'MISSING')}")
        print(f"    long_name:    {ds[var].attrs.get('long_name', 'MISSING')}")
        print(f"    cell_methods: {ds[var].attrs.get('cell_methods', 'MISSING')}")


## Manifest provenance check

In [ ]:
import json as _json

manifest_path = PROJECT / "manifest.json"
if manifest_path.exists():
    manifest = _json.loads(manifest_path.read_text())
    entry = manifest.get("sources", {}).get("era5_land", {})
    print(f"Source key:     {entry.get('source_key', 'N/A')}")
    print(f"Period:         {entry.get('period', 'N/A')}")
    print(f"Variables:      {entry.get('variables', 'N/A')}")
    print(f"Monthly NC:     {entry.get('monthly_nc', 'N/A')}")
    print(f"Years fetched:  {entry.get('years_fetched', 'N/A')}")
    print(f"Timestamp:      {entry.get('download_timestamp', 'N/A')}")
else:
    print(f"manifest.json not found at {manifest_path}")


In [ ]:
ds.close()
